This notebook is used to test the conversion of non-deterministic <span style="font-variant:small-caps;">Fsm</span>s into deterministic <span style="font-variant:small-caps;">Fsm</span>s.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#load "AutomataTypes.fsx"
#load "NfaToDfa.fsx"
#load "FsmToDot.fsx"

In [ ]:
open AutomataTypes
open NfaToDfa
open FsmToDot

The shared F# scripts are exported from the defining notebook cells during setup. Loading them retains static checking. DOT generation does not require Graphviz; install its native executable only for graphical SVG rendering.

The non-deterministic <span style="font-variant:small-caps;">Fsm</span> defined below is taken from the lecture notes.

In [ ]:
let states = [0 .. 7] |> List.map (fun i -> sprintf "q%d" i) |> Set.ofList
let alphabet = set ['a'; 'b']
let delta =
    [ ("q0", Eps), set ["q1"; "q2"]
      ("q1", Input 'b'), set ["q3"]
      ("q2", Input 'a'), set ["q4"]
      ("q3", Input 'a'), set ["q5"]
      ("q4", Input 'b'), set ["q6"]
      ("q5", Eps), set ["q7"]
      ("q6", Eps), set ["q7"]
      ("q7", Eps), set ["q0"] ] |> Map.ofList
let nfa44 : Nfa<string> =
    { States = states; Alphabet = alphabet; Transitions = delta
      Start = "q0"; Accepting = set ["q7"] }

The function `nfa2dot`can be used to render this <span style="font-variant:small-caps;">Fsm</span>.

In [ ]:
let nfaDot = nfa2dot nfa44
printfn "%s" nfaDot

In [ ]:
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(renderSvg nfaDot), "text/html") |> ignore

This recognizes the same language as the language described by
$$ (a \cdot b + b \cdot a) \cdot (a \cdot b + b \cdot a)^* $$
Let us convert it into a deterministic <span style="font-variant:small-caps;">Fsm</span>:

In [ ]:
let dfa44 = nfa2dfa nfa44
let dfaDot, states2Names = dfa2dot dfa44
printfn "%s" dfaDot

In [ ]:
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(renderSvg dfaDot), "text/html") |> ignore

In order to inspect the states of this deterministic <span style="font-variant:small-caps;">Fsm</span> we print the map  `states2Names`.

In [ ]:
states2Names

We can also print the <span style="font-variant:small-caps;">Fsm</span>.

In [ ]:
dfa44

The following words check acceptance and compare the NFA with its deterministic result. The empty word is rejected even though epsilon transitions can cycle back to the start.

In [ ]:
for word, expected in ["", false; "ab", true; "ba", true; "abba", true; "abab", true; "aa", false; "aba", false] do
    checkNotebook (acceptsNfa nfa44 word = expected)
    checkNotebook (acceptsDfa dfa44 word = expected)
checkNotebook (dfa44.Transitions.Count = dfa44.States.Count * alphabet.Count)

An additional NFA in the original notebook recognizes words whose third character from the end is `b`.

In [ ]:
let alternateNfa : Nfa<string> =
    { States = set ["4";"1";"2";"3"]; Alphabet = alphabet
      Transitions = Map.ofList [
          ("4", Input 'a'), set ["4"]
          ("4", Input 'b'), set ["4";"1"]
          ("1", Input 'a'), set ["2"]
          ("1", Input 'b'), set ["2"]
          ("2", Input 'a'), set ["3"]
          ("2", Input 'b'), set ["3"] ]
      Start = "4"; Accepting = set ["3"] }
let alternateDfa = nfa2dfa alternateNfa
for word, expected in ["baa",true; "aba",false; "abaa",true; "ba",false] do
    checkNotebook (acceptsNfa alternateNfa word = expected)
    checkNotebook (acceptsDfa alternateDfa word = expected)